In [63]:
import pandas as pd

# 1. 5.1 불러오기
path_51 = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\annotateAll\dbNSFP5.1_nsSNV.chr1.gz"
df_51 = pd.read_csv(path_51, sep='\t', compression='gzip', low_memory=False, dtype=str)

# 2. 5.2a 불러오기
path_52 = r"E:\CAGI_data\dbNSFP5.2a_grch38_splits\dbNSFP5.2a_1.tsv.gz"
df_52 = pd.read_csv(path_52, sep="\t", compression="gzip", dtype=str, low_memory=False)

# 3. Key 생성 (aapos는 제외)
def make_key(df, chr_col):
    return df[chr_col].astype(str) + "__" + \
           df["pos(1-based)"].astype(str) + "__" + \
           df["ref"].astype(str) + "__" + \
           df["alt"].astype(str) + "__" + \
           df["aaref"].astype(str) + "__" + \
           df["aaalt"].astype(str)

df_51["__key__"] = make_key(df_51, "#chr")
df_52["__key__"] = make_key(df_52, "chr")

# 4. 중복 키 확인 → 로그 저장
dups = df_52[df_52.duplicated("__key__", keep=False)]
if not dups.empty:
    dups.to_csv("log_duplicated_keys_in_52a.tsv", sep="\t", index=False)

# 5. 병합용 데이터 준비 (aapos → mut_pos로 rename)
df_52_slim = df_52[["__key__", "Uniprot_acc", "Uniprot_entry", "aapos"]].copy()
df_52_slim = df_52_slim.rename(columns={"aapos": "mut_pos"})
df_52_slim = df_52_slim.drop_duplicates("__key__")

# 6. 병합
df_merged = df_51.merge(df_52_slim, on="__key__", how="left")

# 7. 매핑 실패 로그 저장
missing = df_merged[df_merged["Uniprot_acc"].isna()]
if not missing.empty:
    missing.to_csv("log_missing_uniprot.tsv", sep="\t", index=False)

# 8. 정리
df_merged.drop(columns="__key__", inplace=True)

# 결과 저장 (옵션)
# df_merged.to_csv("df_51_with_uniprot.tsv", sep="\t", index=False)


In [34]:
df_51

,#chr,pos(1-based),ref,alt,aaref,aaalt,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),genename,cds_strand,refcodon,codonpos,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,aapos,__key__
0,1,65565,A,C,M,L,1,65565,1,55428,OR4F5,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65565__A__C__M__L__1
1,1,65565,A,G,M,V,1,65565,1,55428,OR4F5,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65565__A__G__M__V__1
2,1,65565,A,T,M,L,1,65565,1,55428,OR4F5,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65565__A__T__M__L__1
3,1,65566,T,A,M,K,1,65566,1,55429,OR4F5,+,ATG,2,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65566__T__A__M__K__1
4,1,65566,T,C,M,T,1,65566,1,55429,OR4F5,+,ATG,2,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65566__T__C__M__T__1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8355136,1,248918362,G,C,X,S,1,249212561,1,247179184,PGBD2;PGBD2,+;+,TGA;TGA,2;2,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,1__248918362__G__C__X__S__342;593
8355137,1,248918362,G,T,X,L,1,249212561,1,247179184,PGBD2;PGBD2,+;+,TGA;TGA,2;2,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,1__248918362__G__T__X__L__342;593
8355138,1,248918363,A,C,X,C,1,249212562,1,247179185,PGBD2;PGBD2,+;+,TGA;TGA,3;3,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,1__248918363__A__C__X__C__342;593
8355139,1,248918363,A,G,X,W,1,249212562,1,247179185,PGBD2;PGBD2,+;+,TGA;TGA,3;3,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,1__248918363__A__G__X__W__342;593


In [37]:
df_52

,chr,pos(1-based),ref,alt,aaref,aaalt,rs_dbSNP,hg19_chr,hg19_pos(1-based),hg18_chr,...,TSL,VEP_canonical,MANE,cds_strand,refcodon,codonpos,codon_degeneracy,clinvar_id,clinvar_clnsig,__key__
0,1,65565,A,C,M,L,.,1,65565,1,...,.,YES,Select,+,ATG,1,0,.,.,1__65565__A__C__M__L__1
1,1,65565,A,G,M,V,.,1,65565,1,...,.,YES,Select,+,ATG,1,0,.,.,1__65565__A__G__M__V__1
2,1,65565,A,T,M,L,.,1,65565,1,...,.,YES,Select,+,ATG,1,0,.,.,1__65565__A__T__M__L__1
3,1,65566,T,A,M,K,.,1,65566,1,...,.,YES,Select,+,ATG,2,0,.,.,1__65566__T__A__M__K__1
4,1,65566,T,C,M,T,rs1639927683,1,65566,1,...,.,YES,Select,+,ATG,2,0,.,.,1__65566__T__C__M__T__1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8585110,1,248918362,G,C,X,S,rs1192139239,1,249212561,1,...,1;1,.;YES,.;Select,+;+,TGA;TGA,2;2,2;2,.,.,1__248918362__G__C__X__S__342;593
8585111,1,248918362,G,T,X,L,rs1192139239,1,249212561,1,...,1;1,.;YES,.;Select,+;+,TGA;TGA,2;2,2;2,.,.,1__248918362__G__T__X__L__342;593
8585112,1,248918363,A,C,X,C,.,1,249212562,1,...,1;1,.;YES,.;Select,+;+,TGA;TGA,3;3,0;0,.,.,1__248918363__A__C__X__C__342;593
8585113,1,248918363,A,G,X,W,.,1,249212562,1,...,1;1,.;YES,.;Select,+;+,TGA;TGA,3;3,0;0,.,.,1__248918363__A__G__X__W__342;593


In [56]:
df_52[df_52["pos(1-based)"]=="241683581"]["__key__"]

8331575    1__241683581__T__A__L__X__230;240
8331576          1__241683581__T__A__M__X__1
8331577    1__241683581__T__C__L__S__230;240
8331578          1__241683581__T__C__M__S__1
8331579    1__241683581__T__G__L__W__230;240
8331580          1__241683581__T__G__M__W__1
Name: __key__, dtype: object

In [55]:
df_51[df_51["pos(1-based)"]=="241683581"]["__key__"]

8105465    1__241683581__T__A__L__X__230;240;1
8105466    1__241683581__T__C__L__S__230;240;1
8105467    1__241683581__T__G__L__W__230;240;1
Name: __key__, dtype: object

In [ ]:
missing

,#chr,pos(1-based),ref,alt,aaref,aaalt,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),...,cds_strand,refcodon,codonpos,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,aapos,__key__,Uniprot_acc,Uniprot_entry
858210,1,15716237,T,A,L,M,1,16042732,1,15915319,...,+;+;+,TTG;TTG;TTG,1;1;1,ENSG00000116786;ENSG00000116786;ENSG00000116786,ENST00000375799;ENST00000375793;ENST00000642363,ENSP00000364956;ENSP00000364950;ENSP00000494591,21;21;21,1__15716237__T__A__L__M__21;21;21,NaN,NaN
858211,1,15716237,T,G,L,V,1,16042732,1,15915319,...,+;+;+,TTG;TTG;TTG,1;1;1,ENSG00000116786;ENSG00000116786;ENSG00000116786,ENST00000375799;ENST00000375793;ENST00000642363,ENSP00000364956;ENSP00000364950;ENSP00000494591,21;21;21,1__15716237__T__G__L__V__21;21;21,NaN,NaN
858212,1,15716238,T,A,L,X,1,16042733,1,15915320,...,+;+;+,TTG;TTG;TTG,2;2;2,ENSG00000116786;ENSG00000116786;ENSG00000116786,ENST00000375799;ENST00000375793;ENST00000642363,ENSP00000364956;ENSP00000364950;ENSP00000494591,21;21;21,1__15716238__T__A__L__X__21;21;21,NaN,NaN
858213,1,15716238,T,C,L,S,1,16042733,1,15915320,...,+;+;+,TTG;TTG;TTG,2;2;2,ENSG00000116786;ENSG00000116786;ENSG00000116786,ENST00000375799;ENST00000375793;ENST00000642363,ENSP00000364956;ENSP00000364950;ENSP00000494591,21;21;21,1__15716238__T__C__L__S__21;21;21,NaN,NaN
858214,1,15716238,T,G,L,W,1,16042733,1,15915320,...,+;+;+,TTG;TTG;TTG,2;2;2,ENSG00000116786;ENSG00000116786;ENSG00000116786,ENST00000375799;ENST00000375793;ENST00000642363,ENSP00000364956;ENSP00000364950;ENSP00000494591,21;21;21,1__15716238__T__G__L__W__21;21;21,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8105465,1,241683581,T,A,L,X,1,241846883,1,239913506,...,+;+;+,TTG;TTG;TTG,2;2;2,ENSG00000162843;ENSG00000162843;ENSG00000162843,ENST00000366552;ENST00000437684;ENST00000414635,ENSP00000355510;ENSP00000402446;ENSP00000406656,230;240;1,1__241683581__T__A__L__X__230;240;1,NaN,NaN
8105466,1,241683581,T,C,L,S,1,241846883,1,239913506,...,+;+;+,TTG;TTG;TTG,2;2;2,ENSG00000162843;ENSG00000162843;ENSG00000162843,ENST00000366552;ENST00000437684;ENST00000414635,ENSP00000355510;ENSP00000402446;ENSP00000406656,230;240;1,1__241683581__T__C__L__S__230;240;1,NaN,NaN
8105467,1,241683581,T,G,L,W,1,241846883,1,239913506,...,+;+;+,TTG;TTG;TTG,2;2;2,ENSG00000162843;ENSG00000162843;ENSG00000162843,ENST00000366552;ENST00000437684;ENST00000414635,ENSP00000355510;ENSP00000402446;ENSP00000406656,230;240;1,1__241683581__T__G__L__W__230;240;1,NaN,NaN
8105468,1,241683582,G,C,L,F,1,241846884,1,239913507,...,+;+;+,TTG;TTG;TTG,3;3;3,ENSG00000162843;ENSG00000162843;ENSG00000162843,ENST00000366552;ENST00000437684;ENST00000414635,ENSP00000355510;ENSP00000402446;ENSP00000406656,230;240;1,1__241683582__G__C__L__F__230;240;1,NaN,NaN


In [33]:
df_merged

,#chr,pos(1-based),ref,alt,aaref,aaalt,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),genename,cds_strand,refcodon,codonpos,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,aapos,Uniprot_acc,Uniprot_entry
0,1,65565,A,C,M,L,1,65565,1,55428,OR4F5,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,A0A2U3U0J3,A0A2U3U0J3_HUMAN
1,1,65565,A,G,M,V,1,65565,1,55428,OR4F5,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,A0A2U3U0J3,A0A2U3U0J3_HUMAN
2,1,65565,A,T,M,L,1,65565,1,55428,OR4F5,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,A0A2U3U0J3,A0A2U3U0J3_HUMAN
3,1,65566,T,A,M,K,1,65566,1,55429,OR4F5,+,ATG,2,ENSG00000186092,ENST00000641515,ENSP00000493376,1,A0A2U3U0J3,A0A2U3U0J3_HUMAN
4,1,65566,T,C,M,T,1,65566,1,55429,OR4F5,+,ATG,2,ENSG00000186092,ENST00000641515,ENSP00000493376,1,A0A2U3U0J3,A0A2U3U0J3_HUMAN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8355136,1,248918362,G,C,X,S,1,249212561,1,247179184,PGBD2;PGBD2,+;+,TGA;TGA,2;2,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,Q6P3X8-2;Q6P3X8,PGBD2_HUMAN;PGBD2_HUMAN
8355137,1,248918362,G,T,X,L,1,249212561,1,247179184,PGBD2;PGBD2,+;+,TGA;TGA,2;2,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,Q6P3X8-2;Q6P3X8,PGBD2_HUMAN;PGBD2_HUMAN
8355138,1,248918363,A,C,X,C,1,249212562,1,247179185,PGBD2;PGBD2,+;+,TGA;TGA,3;3,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,Q6P3X8-2;Q6P3X8,PGBD2_HUMAN;PGBD2_HUMAN
8355139,1,248918363,A,G,X,W,1,249212562,1,247179185,PGBD2;PGBD2,+;+,TGA;TGA,3;3,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,Q6P3X8-2;Q6P3X8,PGBD2_HUMAN;PGBD2_HUMAN


In [28]:
path_52 = r"E:\CAGI_data\dbNSFP5.2a_grch38_splits\dbNSFP5.2a_1.tsv.gz"
df_52 = pd.read_csv(path_52, sep="\t", compression="gzip", dtype=str, low_memory=False)

In [29]:
df_52["aapos"], df_52["Uniprot_acc"]

(0                1
 1                1
 2                1
 3                1
 4                1
             ...   
 8585110    342;593
 8585111    342;593
 8585112    342;593
 8585113    342;593
 8585114    342;593
 Name: aapos, Length: 8585115, dtype: object,
 0               A0A2U3U0J3
 1               A0A2U3U0J3
 2               A0A2U3U0J3
 3               A0A2U3U0J3
 4               A0A2U3U0J3
                 ...       
 8585110    Q6P3X8-2;Q6P3X8
 8585111    Q6P3X8-2;Q6P3X8
 8585112    Q6P3X8-2;Q6P3X8
 8585113    Q6P3X8-2;Q6P3X8
 8585114    Q6P3X8-2;Q6P3X8
 Name: Uniprot_acc, Length: 8585115, dtype: object)

In [31]:
df_51["aapos"]

0                1
1                1
2                1
3                1
4                1
            ...   
8355136    342;593
8355137    342;593
8355138    342;593
8355139    342;593
8355140    342;593
Name: aapos, Length: 8355141, dtype: object

In [ ]:
df_52["aapos"], df_52["Uniprot_acc"]

(0          1.0
 1          1.0
 2          1.0
 3          1.0
 4          1.0
           ... 
 8585110    NaN
 8585111    NaN
 8585112    NaN
 8585113    NaN
 8585114    NaN
 Name: aapos, Length: 8585115, dtype: float64,
 0               A0A2U3U0J3
 1               A0A2U3U0J3
 2               A0A2U3U0J3
 3               A0A2U3U0J3
 4               A0A2U3U0J3
                 ...       
 8585110    Q6P3X8-2;Q6P3X8
 8585111    Q6P3X8-2;Q6P3X8
 8585112    Q6P3X8-2;Q6P3X8
 8585113    Q6P3X8-2;Q6P3X8
 8585114    Q6P3X8-2;Q6P3X8
 Name: Uniprot_acc, Length: 8585115, dtype: object)

In [22]:
df_52["__key__"]

0              1__65565__A__C__M__L__1.0
1              1__65565__A__G__M__V__1.0
2              1__65565__A__T__M__L__1.0
3              1__65566__T__A__M__K__1.0
4              1__65566__T__C__M__T__1.0
                       ...              
8585110    1__248918362__G__C__X__S__nan
8585111    1__248918362__G__T__X__L__nan
8585112    1__248918363__A__C__X__C__nan
8585113    1__248918363__A__G__X__W__nan
8585114    1__248918363__A__T__X__C__nan
Name: __key__, Length: 8585115, dtype: object

In [26]:
df_51["__key__"] 

0                    1__65565__A__C__M__L__1
1                    1__65565__A__G__M__V__1
2                    1__65565__A__T__M__L__1
3                    1__65566__T__A__M__K__1
4                    1__65566__T__C__M__T__1
                         ...                
8355136    1__248918362__G__C__X__S__342;593
8355137    1__248918362__G__T__X__L__342;593
8355138    1__248918363__A__C__X__C__342;593
8355139    1__248918363__A__G__X__W__342;593
8355140    1__248918363__A__T__X__C__342;593
Name: __key__, Length: 8355141, dtype: object

In [17]:
missing

,#chr,pos(1-based),ref,alt,aaref,aaalt,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),...,cds_strand,refcodon,codonpos,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,aapos,__key__,Uniprot_acc,Uniprot_entry
0,1,65565,A,C,M,L,1,65565,1,55428,...,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65565__A__C__M__L__1,NaN,NaN
1,1,65565,A,G,M,V,1,65565,1,55428,...,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65565__A__G__M__V__1,NaN,NaN
2,1,65565,A,T,M,L,1,65565,1,55428,...,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65565__A__T__M__L__1,NaN,NaN
3,1,65566,T,A,M,K,1,65566,1,55429,...,+,ATG,2,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65566__T__A__M__K__1,NaN,NaN
4,1,65566,T,C,M,T,1,65566,1,55429,...,+,ATG,2,ENSG00000186092,ENST00000641515,ENSP00000493376,1,1__65566__T__C__M__T__1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8355136,1,248918362,G,C,X,S,1,249212561,1,247179184,...,+;+,TGA;TGA,2;2,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,1__248918362__G__C__X__S__342;593,NaN,NaN
8355137,1,248918362,G,T,X,L,1,249212561,1,247179184,...,+;+,TGA;TGA,2;2,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,1__248918362__G__T__X__L__342;593,NaN,NaN
8355138,1,248918363,A,C,X,C,1,249212562,1,247179185,...,+;+,TGA;TGA,3;3,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,1__248918363__A__C__X__C__342;593,NaN,NaN
8355139,1,248918363,A,G,X,W,1,249212562,1,247179185,...,+;+,TGA;TGA,3;3,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,1__248918363__A__G__X__W__342;593,NaN,NaN


In [16]:
df_merged

,#chr,pos(1-based),ref,alt,aaref,aaalt,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),genename,cds_strand,refcodon,codonpos,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,aapos,Uniprot_acc,Uniprot_entry
0,1,65565,A,C,M,L,1,65565,1,55428,OR4F5,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,NaN,NaN
1,1,65565,A,G,M,V,1,65565,1,55428,OR4F5,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,NaN,NaN
2,1,65565,A,T,M,L,1,65565,1,55428,OR4F5,+,ATG,1,ENSG00000186092,ENST00000641515,ENSP00000493376,1,NaN,NaN
3,1,65566,T,A,M,K,1,65566,1,55429,OR4F5,+,ATG,2,ENSG00000186092,ENST00000641515,ENSP00000493376,1,NaN,NaN
4,1,65566,T,C,M,T,1,65566,1,55429,OR4F5,+,ATG,2,ENSG00000186092,ENST00000641515,ENSP00000493376,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8355136,1,248918362,G,C,X,S,1,249212561,1,247179184,PGBD2;PGBD2,+;+,TGA;TGA,2;2,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,NaN,NaN
8355137,1,248918362,G,T,X,L,1,249212561,1,247179184,PGBD2;PGBD2,+;+,TGA;TGA,2;2,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,NaN,NaN
8355138,1,248918363,A,C,X,C,1,249212562,1,247179185,PGBD2;PGBD2,+;+,TGA;TGA,3;3,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,NaN,NaN
8355139,1,248918363,A,G,X,W,1,249212562,1,247179185,PGBD2;PGBD2,+;+,TGA;TGA,3;3,ENSG00000185220;ENSG00000185220,ENST00000355360;ENST00000329291,ENSP00000355424;ENSP00000331643,342;593,NaN,NaN
